# End-to-End ML Pipeline: CREMA-D Speech Emotion Recognition

This notebook implements a complete machine learning pipeline for speech emotion recognition using the CREMA-D dataset.

## Pipeline Overview:

1. **Data Loading & Preparation** - Load processed features from CSV
2. **Feature Selection** - Select minimal but effective features
3. **Data Upload to S3** - Prepare data for SageMaker training
4. **Model Training** - Train XGBoost classifier on SageMaker
5. **Model Registry** - Register trained model with versioning
6. **Model Deployment** - Deploy endpoint for real-time inference
7. **Testing & Validation** - Test predictions and evaluate performance
8. **Resource Cleanup** - Delete all AWS resources

---

**Dataset:** CREMA-D (Crowd-sourced Emotional Multimodal Actors Dataset)  
**Task:** Multi-class emotion classification (6 emotions)  
**Algorithm:** XGBoost Multi-class Classifier

## Setup and Configuration

Import required libraries and configure AWS credentials following the pattern from Athena_DB_Setup.ipynb

In [14]:
import boto3
import sagemaker
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import time
import os
from datetime import datetime
from sklearn.preprocessing import LabelEncoder

In [15]:
from sagemaker.session import Session
from sagemaker import get_execution_role

# Initialize SageMaker session 
sess = sagemaker.Session()
bucket = sess.default_bucket()
role = sagemaker.get_execution_role()
region = boto3.Session().region_name
account_id = boto3.client("sts").get_caller_identity().get("Account")

# Initialize AWS clients
sagemaker_client = boto3.client("sagemaker", region_name=region)
s3_client = boto3.client("s3")

print("=" * 70)
print("AWS Configuration Initialized")
print("=" * 70)
print(f"SageMaker Role: {role}")
print(f"S3 Bucket: {bucket}")
print(f"Region: {region}")
print(f"Account ID: {account_id}")
print("=" * 70)

ModuleNotFoundError: No module named 'sagemaker.session'

## Load and Prepare Data

Load the processed CREMA-D dataset with audio features and prepare for training.

In [ ]:
# Load processed data
data_path = "artifacts/cremad_processed.csv"
df = pd.read_csv(data_path)

display(df.head(3))

,filepath,speaker_id,sentence_id,emotion,intensity,dataset,language,duration,rms_mean,zcr_mean,...,mfcc_13_mean,mfcc_13_std,chroma_mean,chroma_std,tempo,pitch_mean,pitch_std,pitch_range,voicing_rate,split
0,datasets/CREMAD/AudioWAV/1022_ITS_ANG_XX.wav,1022,ITS,ANG,XX,CREMA-D,en,2.435750,0.143429,0.154541,...,-6.393611,7.658659,0.285081,0.303273,89.285714,263.764600,54.099123,174.396963,0.826923,test
1,datasets/CREMAD/AudioWAV/1037_ITS_ANG_XX.wav,1037,ITS,ANG,XX,CREMA-D,en,3.003000,0.040096,0.119629,...,-3.655557,5.415545,0.360693,0.323950,125.000000,281.845387,82.430371,264.320245,0.627660,train
2,datasets/CREMAD/AudioWAV/1060_ITS_NEU_XX.wav,1060,ITS,NEU,XX,CREMA-D,en,2.402375,0.009055,0.078144,...,-3.776471,4.847745,0.386817,0.300740,170.454545,259.221351,25.534457,97.124785,0.368421,train


## Feature Selection (Minimal Set)

Select the most important features for training. We'll use a minimal set focusing on:
- **Energy**: RMS mean (intensity/loudness)
- **Pitch**: Mean and variability (prosody)
- **Spectral**: Centroid mean (brightness/timbre)
- **MFCCs**: First 3 coefficients (spectral envelope)
- **Rhythm**: Tempo

This gives us **8 features total** instead of 47.

In [ ]:
# Define minimal feature set (8 features)
selected_features = [
    'rms_mean',                    # Energy/intensity
    'pitch_mean',                  # Fundamental frequency
    'pitch_std',                   # Pitch variability
    'spectral_centroid_mean',      # Spectral brightness
    'mfcc_1_mean',                 # MFCC coefficient 1
    'mfcc_2_mean',                 # MFCC coefficient 2
    'mfcc_3_mean',                 # MFCC coefficient 3
    'tempo'                        # Rhythmic tempo
]

# Handle tempo (convert string to numeric)
df['tempo'] = pd.to_numeric(df['tempo'], errors='coerce')

# Encode emotion labels to numeric
label_encoder = LabelEncoder()
df['emotion_encoded'] = label_encoder.fit_transform(df['emotion'])

# Save label mapping
emotion_mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))
print("\n" + "=" * 30)
print("Emotion Label Mapping:")
print("=" * 30)
for emotion, code in emotion_mapping.items():
    print(f"  {emotion}: {code}")

# Create feature matrix
feature_columns = selected_features + ['emotion_encoded']
df_features = df[feature_columns + ['split']].copy()



Emotion Label Mapping:
  ANG: 0
  DIS: 1
  FEA: 2
  HAP: 3
  NEU: 4
  SAD: 5


## Prepare Train/Validation/Test Splits

Split the data according to the existing split column and format for XGBoost (label in first column).

In [ ]:
# Split data according to existing splits
train_data = df_features[df_features['split'] == 'train'].copy()
val_data = df_features[df_features['split'] == 'val'].copy()
test_data = df_features[df_features['split'] == 'test'].copy()

print("=" * 70)
print("Data Split Summary")
print("=" * 70)
print(f"Training samples:   {len(train_data):,} ({len(train_data)/len(df_features)*100:.1f}%)")
print(f"Validation samples: {len(val_data):,} ({len(val_data)/len(df_features)*100:.1f}%)")
print(f"Test samples:       {len(test_data):,} ({len(test_data)/len(df_features)*100:.1f}%)")
print(f"Total samples:      {len(df_features):,}")

# Prepare datasets for XGBoost (label must be first column, no header)
# Format: emotion_encoded, feature1, feature2, ..., feature8
train_xgb = train_data[['emotion_encoded'] + selected_features]
val_xgb = val_data[['emotion_encoded'] + selected_features]
test_xgb = test_data[['emotion_encoded'] + selected_features]

# Save to CSV files (no index, no header for XGBoost)
train_file = 'train_emotion.csv'
val_file = 'validation_emotion.csv'
test_file = 'test_emotion.csv'

train_xgb.to_csv(train_file, index=False, header=False)
val_xgb.to_csv(val_file, index=False, header=False)
test_xgb.to_csv(test_file, index=False, header=False)

print("\n" + "=" * 70)
print("CSV Files Created:")
print("=" * 70)
print(f" {train_file} - {len(train_xgb)} rows x {len(train_xgb.columns)} columns")
print(f" {val_file} - {len(val_xgb)} rows x {len(val_xgb.columns)} columns")
print(f" {test_file} - {len(test_xgb)} rows x {len(test_xgb.columns)} columns")


train_xgb.head()

Data Split Summary
Training samples:   5,148 (69.2%)
Validation samples: 1,147 (15.4%)
Test samples:       1,147 (15.4%)
Total samples:      7,442

CSV Files Created:
✓ train_emotion.csv - 5148 rows x 9 columns
✓ validation_emotion.csv - 1147 rows x 9 columns
✓ test_emotion.csv - 1147 rows x 9 columns

Sample Training Data (first 5 rows):
Format: [emotion_label, rms_mean, pitch_mean, pitch_std, spectral_centroid_mean, mfcc_1, mfcc_2, mfcc_3, tempo]


,emotion_encoded,rms_mean,pitch_mean,pitch_std,spectral_centroid_mean,mfcc_1_mean,mfcc_2_mean,mfcc_3_mean,tempo
1,0,0.040096,281.845387,82.430371,1695.947298,-309.92040,98.228230,34.167892,125.000000
2,4,0.009055,259.221351,25.534457,1482.563695,-381.40660,103.938866,35.128147,170.454545
3,4,0.011636,270.143651,58.167998,1438.130000,-374.81824,108.603020,42.597740,156.250000
5,1,0.010611,155.499702,13.871582,1136.654376,-394.71094,116.956070,42.338467,85.227273
6,5,0.018338,292.625595,86.415581,1170.732349,-378.74990,117.318840,24.074411,144.230769


## Upload Data to S3

Upload training, validation, and test datasets to S3 for SageMaker training.

In [ ]:
# Define S3 prefix for the project
prefix = 'crema-d-emotion-recognition-g2'


train_s3_path = sess.upload_data(
    path=train_file, 
    bucket=bucket,
    key_prefix=f'{prefix}/train'
)
print(f" Training data uploaded to:   {train_s3_path}")


val_s3_path = sess.upload_data(
    path=val_file, 
    bucket=bucket,
    key_prefix=f'{prefix}/validation'
)
print(f" Validation data uploaded to: {val_s3_path}")


test_s3_path = sess.upload_data(
    path=test_file, 
    bucket=bucket,
    key_prefix=f'{prefix}/test'
)
print(f" Test data uploaded to:       {test_s3_path}")

# Store paths for later use
s3_output_location = f's3://{bucket}/{prefix}/output'
print(f"\n Model output will be saved to: {s3_output_location}")


In [ ]:
# Delete temporary local files
print("\nDeleting temp Local Files...")
local_files = [train_file, val_file, test_file]
for file in local_files:
    try:
        if os.path.exists(file):
            os.remove(file)
            print(f"    Deleted {file}")
    except Exception as e:
        print(f"   Error deleting {file}: {e}")




Deleting temp Local Files...
   ✓ Deleted train_emotion.csv
   ✓ Deleted validation_emotion.csv
   ✓ Deleted test_emotion.csv


## Configure Training Job

Set up XGBoost estimator with hyperparameters optimized for multi-class emotion classification.

In [ ]:
# Create unique training job name
job_name = f'crema-d-emotion-xgboost-{datetime.now().strftime("%Y-%m-%d-%H-%M-%S")}'

# Get XGBoost container image
container = sagemaker.image_uris.retrieve(
    framework='xgboost',
    region=region,
    version='1.7-1'
)

# Create estimator
xgb_estimator = sagemaker.estimator.Estimator(
    image_uri=container,
    role=role,
    instance_count=1,
    instance_type='ml.m5.xlarge',
    volume_size=50,
    max_run=3600,
    input_mode='File',
    output_path=s3_output_location,
    sagemaker_session=sess,
    base_job_name='crema-d-emotion-xgboost'
)

# Set hyperparameters for multi-class classification
# num_class = 6 emotions (ANG, DIS, FEA, HAP, NEU, SAD)
num_classes = len(df['emotion'].unique())

xgb_estimator.set_hyperparameters(
    objective='multi:softmax',      # Multi-class classification
    num_class=num_classes,          # Number of emotion classes
    max_depth=6,                    # Tree depth
    eta=0.3,                        # Learning rate
    gamma=0,                        # Minimum loss reduction
    min_child_weight=1,             # Minimum sum of instance weight
    subsample=0.8,                  # Subsample ratio
    colsample_bytree=0.8,           # Feature subsample ratio
    eval_metric='mlogloss',         # Multi-class log loss
    num_round=100                   # Number of boosting rounds
)


## Train the Model

Start the training job with training and validation data channels.

In [ ]:
# Define input data channels
train_input = sagemaker.inputs.TrainingInput(
    s3_data=train_s3_path,
    content_type='text/csv',
    s3_data_type='S3Prefix'
)

validation_input = sagemaker.inputs.TrainingInput(
    s3_data=val_s3_path,
    content_type='text/csv',
    s3_data_type='S3Prefix'
)

data_channels = {
    'train': train_input,
    'validation': validation_input
}

print("=" * 30)
print("Starting Training Job")
print("=" * 30)
print(f"Training data: {train_s3_path}")
print(f"Validation data: {val_s3_path}")
print(f"\nThis will take a few minutes...")
print("=" * 30)

# Start training
xgb_estimator.fit(
    inputs=data_channels,
    job_name=job_name,
    logs=True
)


print(" Training Complete!")


## Analyze Training Results

Retrieve and display training metrics and model artifacts.

In [ ]:
# Get training job information
training_job_info = sagemaker_client.describe_training_job(TrainingJobName=job_name)

print("=" * 30)
print("Training Job Results")
print("=" * 30)
print(f"""Job Name: {training_job_info['TrainingJobName']}
    Status: {training_job_info['TrainingJobStatus']}
    Training Time: {training_job_info['TrainingTimeInSeconds']} seconds
    Billable Time: {training_job_info['BillableTimeInSeconds']} seconds""")


model_artifacts = training_job_info['ModelArtifacts']['S3ModelArtifacts']
print(f"\nModel Artifacts: {model_artifacts}")


if 'FinalMetricDataList' in training_job_info:
    print("\n" + "=" * 30)
    print("Training Metrics:")
    print("=" * 30)
    for metric in training_job_info['FinalMetricDataList']:
        print(f"  {metric['MetricName']}: {metric['Value']:.6f}")


# Store for later use
model_data_url = model_artifacts

## Register Model to Model Registry

Create a Model Package Group and register the trained model with versioning.

In [ ]:
# Define model package group
model_package_group_name = 'crema-d-emotion-recognition-models'


try:
    # Create model package group
    create_mpg_response = sagemaker_client.create_model_package_group(
        ModelPackageGroupName=model_package_group_name,
        ModelPackageGroupDescription=(
            'Model Package Group for CREMA-D Speech Emotion Recognition. '
            'Contains XGBoost models trained on audio features to classify '
            'emotions (Anger, Disgust, Fear, Happiness, Neutral, Sadness).'
        )
    )
    print(f" Model Package Group Created")
    print(f"  ARN: {create_mpg_response['ModelPackageGroupArn']}")
except sagemaker_client.exceptions.ResourceInUse:
    print(f" Model Package Group '{model_package_group_name}' already exists")

# Get the model package group details
mpg_details = sagemaker_client.describe_model_package_group(
    ModelPackageGroupName=model_package_group_name
)
print(f"\nModel Package Group: {mpg_details['ModelPackageGroupName']}")
print(f"Status: {mpg_details['ModelPackageGroupStatus']}")

### Create Model Package with Inference Specification

In [ ]:
# Create Model Package
model_package_description = (
    f'XGBoost Emotion Classifier v1.0 - Trained on CREMA-D dataset with {len(selected_features)} features. '
    f'Multi-class classifier for {num_classes} emotions. '
    f'Training job: {job_name}'
)

inference_specification = {
    'Containers': [
        {
            'Image': container,
            'ModelDataUrl': model_data_url,
            'Framework': 'XGBOOST',
            'FrameworkVersion': '1.7-1'
        }
    ],
    'SupportedContentTypes': ['text/csv', 'application/json'],
    'SupportedResponseMIMETypes': ['text/csv', 'application/json'],
    'SupportedRealtimeInferenceInstanceTypes': [
        'ml.t2.medium',
        'ml.t2.large', 
        'ml.m5.large',
        'ml.m5.xlarge'
    ],
    'SupportedTransformInstanceTypes': [
        'ml.m5.large',
        'ml.m5.xlarge',
        'ml.m5.2xlarge'
    ]
}


create_model_package_response = sagemaker_client.create_model_package(
    ModelPackageGroupName=model_package_group_name,
    ModelPackageDescription=model_package_description,
    InferenceSpecification=inference_specification,
    ModelApprovalStatus='Approved'  # Auto-approve for deployment
)

model_package_arn = create_model_package_response['ModelPackageArn']

print(f" Model Package Created")
print(f"  ARN: {model_package_arn}")

# Wait for model package to be ready
print("\nWaiting for model package to be ready...")
while True:
    mp_status = sagemaker_client.describe_model_package(ModelPackageName=model_package_arn)
    status = mp_status['ModelPackageStatus']
    if status in ['Completed', 'Failed']:
        break
    print(f"  Status: {status}...")
    time.sleep(10)

print(f" Model Package Status: {status}")

### Create Model Card with Documentation

In [ ]:
# Create comprehensive Model Card
model_card_name = 'crema-d-emotion-recognition-card'

model_card_content = {
    'model_overview': {
        'model_description': (
            'XGBoost-based multi-class emotion classification model for speech emotion recognition. '
            'The model analyzes 8 key audio features extracted from speech signals to classify '
            'utterances into 6 emotion categories: Anger (ANG), Disgust (DIS), Fear (FEA), '
            'Happiness (HAP), Neutral (NEU), and Sadness (SAD).'
        ),
        'model_owner': 'AAI-540 Final Project - Group 2',
        'model_creator': 'Victor, Donavan, Robair',
        'problem_type': 'Multi-class Classification',
        'algorithm_type': 'XGBoost',
        'model_id': model_package_arn,
        'model_version': '1.0'
    },
    'intended_uses': {
        'purpose_of_model': (
            'This model is designed for automatic speech emotion recognition in conversational AI systems, '
            'call center analytics, mental health monitoring, and human-computer interaction applications. '
            'It processes audio features to determine the emotional state of speakers.'
        ),
        'out_of_scope_use_cases': (
            'NOT intended for clinical diagnosis of mental health conditions. Should not be used as the sole '
            'basis for critical decisions affecting individuals. Not validated for languages other than English '
            'or audio quality significantly different from the training data.'
        )
    },
    'training_details': {
        'objective_function': {
            'function': 'multi:softmax',
            'notes': f'Multi-class classification with {num_classes} emotion classes'
        },
        'training_observations': (
            f'Model trained with XGBoost 1.7-1 on AWS SageMaker ml.m5.xlarge instances. '
            f'Training samples: {len(train_data)}, Validation samples: {len(val_data)}. '
            f'Features used: {", ".join(selected_features)}. '
            'Hyperparameters: max_depth=6, eta=0.3, subsample=0.8, colsample_bytree=0.8, num_round=100.'
        ),
        'training_job_details': {
            'training_arn': training_job_info['TrainingJobArn'],
            'training_datasets': [train_s3_path],
            'training_environment': {
                'container_image': [container]
            }
        }
    },
    'evaluation_details': [
        {
            'name': 'Validation Set Evaluation',
            'evaluation_observation': (
                'Model evaluated on held-out validation set from CREMA-D dataset. '
                'Performance measured using multi-class log loss metric.'
            ),
            'datasets': [val_s3_path],
            'metadata': {
                'dataset_type': 'validation',
                'dataset_name': 'CREMA-D',
                'num_samples': len(val_data)
            }
        }
    ],
    'additional_information': {
        'ethical_considerations': (
            'Emotion recognition models may have biases based on training data demographics. '
            'Performance may vary across different accents, ages, and cultural backgrounds. '
            'Should be used with human oversight in sensitive applications.'
        ),
        'caveats_and_recommendations': (
            'Model trained on acted emotions from CREMA-D dataset. Real-world spontaneous emotions '
            'may differ in acoustic characteristics. Regular monitoring and retraining recommended. '
            'Best performance expected with clear audio similar to training conditions.'
        )
    }
}

print("=" * 30)
print("Creating Model Card")
print("=" * 30)

try:
    create_model_card_response = sagemaker_client.create_model_card(
        ModelCardName=model_card_name,
        Content=json.dumps(model_card_content),
        ModelCardStatus='Draft',
        Tags=[
            {'Key': 'Project', 'Value': 'AAI540-FinalProject'},
            {'Key': 'Team', 'Value': 'Group2'},
            {'Key': 'ModelType', 'Value': 'XGBoost-MultiClass'}
        ]
    )
    print(f" Model Card Created")
    print(f"  ARN: {create_model_card_response['ModelCardArn']}")
except sagemaker_client.exceptions.ResourceInUse:
    print(f" Model Card '{model_card_name}' already exists, updating...")
    update_response = sagemaker_client.update_model_card(
        ModelCardName=model_card_name,
        Content=json.dumps(model_card_content),
        ModelCardStatus='Draft'
    )
    print(f"  ARN: {update_response['ModelCardArn']}")

print("\n Model registered successfully to Model Registry")

## Deploy Model to Endpoint

Create and deploy a real-time inference endpoint.

In [ ]:
# Define endpoint name
endpoint_name = f'crema-d-emotion-endpoint-{datetime.now().strftime("%Y-%m-%d-%H-%M")}'

# Deploy the model from the estimator
predictor = xgb_estimator.deploy(
    initial_instance_count=1,
    instance_type='ml.t2.medium',
    endpoint_name=endpoint_name,
    serializer=sagemaker.serializers.CSVSerializer(),
    deserializer=sagemaker.deserializers.CSVDeserializer()
)

print("\n" + "=" * 30)
print(" Model Deployed Successfully!")
print("=" * 30)
print(f"Endpoint Name: {endpoint_name}")
print(f"Endpoint ARN: {predictor.endpoint_arn}")
print("=" * 30)

##  Test Model Predictions

Test the deployed endpoint with sample data from the test set.

In [ ]:
# Get test samples (without labels)
test_samples = test_xgb[selected_features].head(10)


print(f"Number of test samples: {len(test_samples)}")
print("\nSample features (first 3):")
display(test_samples.head(3))

# Get actual labels
actual_labels = test_xgb['emotion_encoded'].head(10).values
actual_emotions = [list(emotion_mapping.keys())[list(emotion_mapping.values()).index(label)] 
                   for label in actual_labels]

# Make predictions
predictions = predictor.predict(test_samples.values)

# Parse predictions (they come as list of lists)
predicted_labels = [int(float(pred[0])) for pred in predictions]
predicted_emotions = [list(emotion_mapping.keys())[list(emotion_mapping.values()).index(label)] 
                      for label in predicted_labels]

# Display results
results_df = pd.DataFrame({
    'Sample': range(1, len(test_samples) + 1),
    'Actual_Emotion': actual_emotions,
    'Predicted_Emotion': predicted_emotions,
    'Correct': [actual == pred for actual, pred in zip(actual_emotions, predicted_emotions)]
})

print("Prediction Results:")
print("=" * 30)
display(results_df)

# Calculate accuracy
accuracy = results_df['Correct'].sum() / len(results_df)
print(f"\nAccuracy on sample: {accuracy*100:.1f}% ({results_df['Correct'].sum()}/{len(results_df)} correct)")


### Batch Prediction Test

Test with a larger batch to get more comprehensive evaluation.

In [ ]:
# Test with larger batch (100 samples)
batch_size = min(100, len(test_xgb))
test_batch = test_xgb[selected_features].head(batch_size)
actual_batch_labels = test_xgb['emotion_encoded'].head(batch_size).values

print(f"Batch Prediction Test ({batch_size} samples)")

# Make batch predictions
batch_predictions = predictor.predict(test_batch.values)
predicted_batch_labels = [int(float(pred[0])) for pred in batch_predictions]

# Calculate metrics
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

accuracy = accuracy_score(actual_batch_labels, predicted_batch_labels)
print(f"\nOverall Accuracy: {accuracy*100:.2f}%")

# Classification report
emotion_names = list(emotion_mapping.keys())
print("\n" + "=" * 30)
print("Classification Report:")
print("=" * 30)
print(classification_report(actual_batch_labels, predicted_batch_labels, 
                          target_names=emotion_names, zero_division=0))

# Confusion matrix
print("=" * 30)
print("Confusion Matrix:")
print("=" * 30)
cm = confusion_matrix(actual_batch_labels, predicted_batch_labels)
cm_df = pd.DataFrame(cm, index=emotion_names, columns=emotion_names)
print(cm_df)

# Visualize confusion matrix
plt.figure(figsize=(10, 8))
sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues', cbar=True)
plt.title('Confusion Matrix - Emotion Classification\n(Test Set Predictions)', 
          fontsize=14, fontweight='bold')
plt.ylabel('Actual Emotion', fontsize=12, fontweight='bold')
plt.xlabel('Predicted Emotion', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrix_test.png', dpi=300, bbox_inches='tight')
plt.show()


## Resource Cleanup

**IMPORTANT: SAVE MONEY** Run this section to delete all AWS resources and avoid charges.

This will delete:
- Endpoint
- Endpoint Configuration
- Model
- Model Package
- Model Package Group  
- Model Card
- S3 Data
- Local CSV files

In [ ]:
import os

print("=" * 70)
print("RESOURCE CLEANUP - Deleting All AWS Resources")
print("=" * 70)
print("\n⚠️  This will delete all resources created in this notebook")
print("⚠️  Proceed with caution!\n")

predictor.delete_endpoint()
print(f"   Endpoint '{endpoint_name}' deleted")

# Wait a moment for endpoint deletion to propagate
time.sleep(5)

sagemaker_client.delete_endpoint_config(EndpointConfigName=endpoint_name)
print(f"   Endpoint config '{endpoint_name}' deleted")

model_name = xgb_estimator.latest_training_job.name
sagemaker_client.delete_model(ModelName=model_name)
print(f"   Model '{model_name}' deleted")

sagemaker_client.delete_model_card(ModelCardName=model_card_name)
print(f"   Model Card '{model_card_name}' deleted")

sagemaker_client.delete_model_package(ModelPackageName=model_package_arn)
print(f"   Model Package deleted")

# Wait for model package deletion
time.sleep(5)

sagemaker_client.delete_model_package_group(ModelPackageGroupName=model_package_group_name)
print(f"   Model Package Group '{model_package_group_name}' deleted")

# List and delete all objects in the prefix
paginator = s3_client.get_paginator('list_objects_v2')
pages = paginator.paginate(Bucket=bucket, Prefix=prefix)

delete_count = 0
for page in pages:
    if 'Contents' in page:
        objects = [{'Key': obj['Key']} for obj in page['Contents']]
        if objects:
            s3_client.delete_objects(Bucket=bucket, Delete={'Objects': objects})
            delete_count += len(objects)

print(f"   Deleted {delete_count} objects from s3://{bucket}/{prefix}/")

print("\nDeleting Local Files...")
local_files = [train_file, val_file, test_file, 'confusion_matrix_test.png']
for file in local_files:
    try:
        if os.path.exists(file):
            os.remove(file)
            print(f"    Deleted {file}")
    except Exception as e:
        print(f"   Error deleting {file}: {e}")

print("\n" + "=" * 30)
print(" CLEANUP COMPLETE!")
print("=" * 30)
print("\nAll resources have been deleted.")
print("\nRemaining manual checks (if needed):")
print("  • CloudWatch Logs - delete log groups if desired")
print("  • S3 Bucket - verify bucket is empty")
print("  • IAM Roles - review if custom roles need deletion")
print("\n" + "=" * 30)

---

## Pipeline Summary

### ✅ Completed Steps:

1. **Setup & Configuration** - Initialized AWS SageMaker and S3 resources
2. **Data Loading** - Loaded CREMA-D processed dataset (7,442 samples)
3. **Feature Selection** - Selected 8 minimal but effective audio features
4. **Data Splitting** - Prepared train/val/test splits
5. **S3 Upload** - Uploaded datasets to S3 for SageMaker training
6. **Training Job** - Trained XGBoost multi-class classifier
7. **Results Analysis** - Evaluated training metrics
8. **Model Registry** - Registered model with versioning
9. **Model Card** - Created comprehensive model documentation
10. **Deployment** - Deployed model to real-time endpoint
11. **Testing** - Validated predictions with test data
12. **Cleanup** - Deleted all AWS resources

### 📊 Model Specifications:

- **Algorithm:** XGBoost 1.7-1 (Multi-class Softmax)
- **Classes:** 6 emotions (ANG, DIS, FEA, HAP, NEU, SAD)
- **Features:** 8 audio features (energy, pitch, spectral, MFCCs, tempo)
- **Training Samples:** ~5,954 (80%)
- **Validation Samples:** ~744 (10%)
- **Test Samples:** ~744 (10%)

### 🎯 Key Features Used:

1. `rms_mean` - Energy/intensity
2. `pitch_mean` - Fundamental frequency  
3. `pitch_std` - Pitch variability
4. `spectral_centroid_mean` - Spectral brightness
5. `mfcc_1_mean` - MFCC coefficient 1
6. `mfcc_2_mean` - MFCC coefficient 2
7. `mfcc_3_mean` - MFCC coefficient 3
8. `tempo` - Rhythmic tempo

### 📝 Notes:

- Model trained on AWS SageMaker ml.m5.xlarge instance
- Endpoint deployed on ml.t2.medium for cost efficiency
- All resources can be recreated by running cells sequentially
- Remember to run cleanup cell to avoid ongoing charges

---

**End of Pipeline**